# Lab 2 — Worker pentru fluxul de predicții

Notebook pornit de `lab2_databricks` (secțiunea 3.3) cu `dbutils.notebook.run`. Citește `stream_incoming` cu Structured Streaming, aplică modelul salvat și face MERGE în `gold_predictions_stream`. Progresul este păstrat în checkpoint.

Parametri:
- `run_label` — eticheta rulării, folosită în jurnalul de monitorizare;
- `crash_after_batch` — dacă este completat, job-ul cade după ce lotul respectiv a fost scris în tabel, dar înainte de commit-ul din checkpoint (cădere simulată).

In [0]:
import json
import time

from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

dbutils.widgets.text("run_label", "manual")
dbutils.widgets.text("crash_after_batch", "")
RUN_LABEL = dbutils.widgets.get("run_label")
CRASH_AFTER_BATCH = (int(dbutils.widgets.get("crash_after_batch"))
                     if dbutils.widgets.get("crash_after_batch").strip() else None)

CATALOG, SCHEMA = "workspace", "lab2"
ART = f"/Volumes/{CATALOG}/{SCHEMA}/artifacts"
MODEL_PATH = f"{ART}/models/ot_classifier"
CHECKPOINT = f"{ART}/checkpoints/predictions_stream"
STREAM_SOURCE = f"{CATALOG}.{SCHEMA}.stream_incoming"
STREAM_SINK = f"{CATALOG}.{SCHEMA}.gold_predictions_stream"
MONITOR_LOG = f"{CATALOG}.{SCHEMA}.pipeline_monitoring_log"

MODEL_VERSION = (spark.table(f"{CATALOG}.{SCHEMA}.gold_model_metrics")
                 .orderBy(F.desc("evaluated_at")).first()["model_version"])
print(f"Rulare: {RUN_LABEL} | cădere simulată după lotul: {CRASH_AFTER_BATCH} | model: {MODEL_VERSION}")

In [0]:
def add_prediction_columns(scored, model_version):
    return (scored
        .withColumn("ot_probability", F.round(vector_to_array("probability")[1], 4))
        .withColumn("prediction", F.col("prediction").cast("int"))
        .withColumn("label", F.col("label").cast("int"))
        .withColumn("model_version", F.lit(model_version))
        .withColumn("scored_at", F.current_timestamp())
        .select("record_id", "fiscal_year", "agency_name", "work_location_borough",
                "pay_basis", "title_group", "base_salary", "ot_hours", "total_ot_paid",
                "label", "prediction", "ot_probability", "model_version", "scored_at"))


def log_event(session, batch_id, status, rows=0, inserted=0, duration=0.0, error=None):
    (session.createDataFrame(
        [(RUN_LABEL, int(batch_id), status, int(rows), int(inserted), float(duration), error)],
        "run_label string, batch_id bigint, status string, input_rows bigint, "
        "inserted_rows bigint, duration_s double, error string")
     .withColumn("logged_at", F.current_timestamp())
     .write.format("delta").mode("append").saveAsTable(MONITOR_LOG))


def score_and_merge(batch_df, batch_id):
    session = batch_df.sparkSession
    started = time.time()
    rows = batch_df.count()
    model = PipelineModel.load(MODEL_PATH)
    scored = add_prediction_columns(model.transform(batch_df), MODEL_VERSION)
    view = f"updates_{RUN_LABEL}_{batch_id}"
    scored.createOrReplaceTempView(view)
    merge_result = session.sql(f'''
        MERGE INTO {STREAM_SINK} AS target
        USING {view} AS source
        ON target.record_id = source.record_id AND target.fiscal_year = source.fiscal_year
        WHEN NOT MATCHED THEN INSERT *
    ''').first()
    inserted = int(merge_result["num_inserted_rows"])
    duration = round(time.time() - started, 2)
    if CRASH_AFTER_BATCH is not None and batch_id == CRASH_AFTER_BATCH:
        message = f"Cădere simulată după scrierea lotului {batch_id}, înainte de commit"
        log_event(session, batch_id, "FAILED", rows, inserted, duration, message)
        raise RuntimeError(message)
    log_event(session, batch_id, "OK", rows, inserted, duration)

In [0]:
query = (spark.readStream.option("maxFilesPerTrigger", 1).table(STREAM_SOURCE)
         .writeStream
         .foreachBatch(score_and_merge)
         .option("checkpointLocation", CHECKPOINT)
         .trigger(availableNow=True)
         .queryName(f"predictions_{RUN_LABEL}")
         .start())
query.awaitTermination()

progress = [p if isinstance(p, dict) else json.loads(p.json) for p in query.recentProgress]


def input_rows(p):
    return int(p.get("numInputRows")
               or sum(s.get("numInputRows") or 0 for s in p.get("sources") or []))


def rows_per_second(p):
    return float(round(p.get("processedRowsPerSecond")
                       or sum(s.get("processedRowsPerSecond") or 0 for s in p.get("sources") or []), 1))


progress_rows = [{
    "batch_id": int(p["batchId"]),
    "input_rows": input_rows(p),
    "duration_ms": int((p.get("durationMs") or {}).get("triggerExecution") or 0),
    "rows_per_s": rows_per_second(p),
} for p in progress]
print(progress_rows)
dbutils.notebook.exit(json.dumps({"run_label": RUN_LABEL, "progress": progress_rows}))